# Project 5: have Paramaribo's hot days become hotter?

**The question.** How did the daily maximum temperature in Paramaribo in 2016 to 2025 differ from 1991 to 2000? Compare the two decades by their averages, their spread, their shape and their share of hot days, and decide whether hot days have become hotter and more common.

**The data.** The daily maximum air temperature at 2 metres above the ground, for every day from 1991 to 2025, from Open-Meteo's historical weather API. The request asks for the ERA5 reanalysis only. A reanalysis is a weather model that combines observations from weather stations, aircraft, buoys, radar and satellites, and estimates the weather on a grid. Each value therefore describes a grid cell about 25 km across, not one thermometer. For the point 5.85 N, 55.20 W, the API answers with the grid cell centred at 5.75 N, 55.25 W. The table was downloaded on 2026-11-06.

**Your team.** Four roles rotate every week: builder, data checker, AI checker and presenter. This week the data checker checks that every share in the note is a count of days divided by the right number of days.

**What you hand in on Friday.** This notebook with every CHECK passing, two charts, a note of about 200 words for a ministry, and your AI log.

## Load the data

This cell is given. It reads the table from the course folder, or straight from Open-Meteo if the file is not there. The table has two columns: `date` and `tmax_c`, the day's maximum temperature in degrees Celsius.

In [ ]:
import json, urllib.request
import pandas as pd

URL = ("https://archive-api.open-meteo.com/v1/archive?latitude=5.85&longitude=-55.20"
       "&start_date=1991-01-01&end_date=2025-12-31&daily=temperature_2m_max"
       "&timezone=America%2FParamaribo&models=era5")
try:
    temps = pd.read_csv("https://santerre.ai/bigi-data/data/week05-paramaribo-tmax-1991-2025.csv", parse_dates=["date"])
except FileNotFoundError:
    raw = json.load(urllib.request.urlopen(URL))["daily"]
    temps = pd.DataFrame({"date": pd.to_datetime(raw["time"]), "tmax_c": raw["temperature_2m_max"]})
temps

## TODO 1: describe the table

Write code that prints the number of rows, the first and last date, the number of missing values, and the lowest and highest temperature with the date of each.

In [ ]:
# TODO 1

## TODO 2: the two decades and their centres

Store in `early` the temperatures from 1 January 1991 to 31 December 2000, and in `recent` those from 1 January 2016 to 31 December 2025. Then store the mean and the median of each decade in `mean_early`, `mean_recent`, `median_early` and `median_recent`.

Count the days in each decade with code. Do not type the number of days.

In [ ]:
# TODO 2
early = None
recent = None
mean_early = mean_recent = median_early = median_recent = None

In [ ]:
# CHECK 2
yr = temps["date"].dt.year
e_ = temps.loc[yr.between(1991, 2000), "tmax_c"]
r_ = temps.loc[yr.between(2016, 2025), "tmax_c"]
assert early is not None and recent is not None, "early or recent is still empty"
assert len(early) == len(pd.date_range("1991-01-01", "2000-12-31")), "early does not hold every day of 1991 to 2000"
assert len(recent) == len(pd.date_range("2016-01-01", "2025-12-31")), "recent does not hold every day of 2016 to 2025"
assert abs(mean_early - e_.mean()) < 0.01 and abs(mean_recent - r_.mean()) < 0.01, "check the means"
assert abs(median_early - e_.median()) < 0.01 and abs(median_recent - r_.median()) < 0.01, "check the medians"
print("CHECK 2 passed:", len(early), "and", len(recent), "days")

## TODO 3: the spread

Make a table called `spread` with one row per decade (the index is `"1991-2000"` and `"2016-2025"`) and three columns: `sd` (the standard deviation), `p10` and `p90` (the 10th and 90th percentiles). Then add a column `p90_minus_p10`.

In [ ]:
# TODO 3
spread = None

In [ ]:
# CHECK 3
assert spread is not None, "spread is still empty"
for label, s in [("1991-2000", e_), ("2016-2025", r_)]:
    assert abs(spread.loc[label, "sd"] - s.std()) < 0.01, f"sd for {label}"
    assert abs(spread.loc[label, "p10"] - s.quantile(0.1)) < 0.05, f"p10 for {label}"
    assert abs(spread.loc[label, "p90"] - s.quantile(0.9)) < 0.05, f"p90 for {label}"
    assert abs(spread.loc[label, "p90_minus_p10"] - (s.quantile(0.9) - s.quantile(0.1))) < 0.1, f"p90_minus_p10 for {label}"
print("CHECK 3 passed")

## TODO 4: draw both decades

Draw one histogram with both decades on the same bins (bins of 0.5 °C work well). Show the share of days on the vertical axis, not the count, so the two decades are drawn on the same footing. Give the chart a title, label both axes, and name Open-Meteo and ERA5 as the source.

In [ ]:
# TODO 4

## TODO 5: hot days

Call a day hot when it was hotter than 90% of the days in 1991 to 2000. Store that temperature, the 90th percentile of `early`, in `threshold`. Then store in `share_early` and `share_recent` the share of days in each decade that were strictly above `threshold`, as a fraction between 0 and 1.

The threshold comes from the early decade only, so both decades are measured against the same older normal.

In [ ]:
# TODO 5
threshold = None
share_early = share_recent = None

In [ ]:
# CHECK 5
t_ = e_.quantile(0.9)
assert threshold is not None and share_early is not None and share_recent is not None, "a value is still empty"
assert abs(threshold - t_) < 0.05, "the threshold is the 90th percentile of 1991 to 2000"
assert 0 <= share_early <= 1 and 0 <= share_recent <= 1, "give the shares as fractions between 0 and 1"
assert abs(share_early - (e_ > t_).mean()) < 0.001 and abs(share_recent - (r_ > t_).mean()) < 0.001, "count the days strictly above the threshold"
print("CHECK 5 passed:", threshold, "C")

## TODO 6: the AI check

Ask your AI assistant this question, with only your two averages in it:

*"The average daily maximum temperature in Paramaribo was A °C in 1991 to 2000 and B °C in 2016 to 2025. Is that a large change or a small one? Have hot days become more common, and by how much?"*

Replace A and B with your `mean_early` and `mean_recent`, rounded to one decimal. Write its answer in your AI log.

Then store in `times_more` how many times as common hot days were in 2016 to 2025 as in 1991 to 2000. In the AI log, say whether the assistant's answer matches the data and what the two averages hid.

In [ ]:
# TODO 6
times_more = None

In [ ]:
# CHECK 6
assert times_more is not None, "times_more is still empty"
assert abs(times_more - (r_ > t_).mean() / (e_ > t_).mean()) < 0.01, "divide the recent share by the early share"
print("CHECK 6 passed:", round(times_more, 2))

## TODO 7: month by month

One average for the whole year can hide a season. Make a table called `by_month` with one row per month (the index is 1 to 12) and three columns: `early` and `recent` (the mean daily maximum for that month in each decade) and `change` (recent minus early). Store in `biggest_month` the number of the month with the largest rise. Draw a bar chart of `change` by month, with a title and labelled axes.

In [ ]:
# TODO 7
by_month = None
biggest_month = None

In [ ]:
# CHECK 7
m_ = temps["date"].dt.month
expected = (temps[yr.between(2016, 2025)].groupby(m_)["tmax_c"].mean()
            - temps[yr.between(1991, 2000)].groupby(m_)["tmax_c"].mean())
assert by_month is not None and biggest_month is not None, "by_month or biggest_month is still empty"
assert len(by_month) == 12, "by_month needs one row per month"
assert abs(by_month["change"].values - expected.values).max() < 0.01, "check the change column"
assert int(biggest_month) == int(expected.idxmax()), "biggest_month is the month with the largest rise"
print("CHECK 7 passed: the largest rise is in month", int(biggest_month))

## TODO 8: the note for a ministry

In a text cell below, write about 200 words for an official at a ministry who plans for heat in schools and clinics. Use the numbers you computed, and report at least one measure of spread and the share of hot days, not only the averages. Name the source, and say that each value is a reanalysis grid cell rather than a weather station. Say one thing the data cannot tell the official.